In [25]:
import pandas as pd
from tqdm import tqdm  # Import tqdm for progress bar visualization in loops
import traceback  # Import traceback for detailed error printing
from openai import OpenAI  # Import OpenAI client to call the LLM API
import re  # Import regex module (NOTE: required because you use re.search later)
import time

client = OpenAI(api_key = "api_key_here")  # Initialize OpenAI client with API key (replace "x" with real key)
model = "gpt-4o-mini"  # Define which OpenAI model to use for evaluation

# Define a large prompt template used to evaluate tax answers
JUDGE_PROMPT = """
You are an expert Austrian tax law evaluator.

Your task is to assess how accurately and correctly a system-generated answer responds to a tax law question under Austrian law.

You MUST evaluate strictly based on Austrian tax regulations (e.g. EStG, UStG, BAO etc.). Do not assume general tax knowledge is sufficient.

Inputs:
- TAX QUESTION
- REFERENCE ANSWER (legally correct answer)
- SYSTEM ANSWER (to be evaluated)

Evaluation criteria:

1. Legal correctness
- Must comply with Austrian tax law
- Check tax rates, thresholds, and legal rules

2. Jurisdiction accuracy
- Must be Austria-specific (not Germany or generic EU law)

3. Completeness
- Compare against reference answer coverage

4. Risk of misinformation
- Penalize hallucinated or incorrect legal claims heavily

Grading Scale (ONLY 1–4):

1 — Incorrect / Dangerous
- Legally wrong under Austrian tax law
- Misleading or hallucinated rules
- Wrong jurisdiction or tax rates
- Would lead to incorrect tax decisions

2 — Partially correct
- Some correct elements but important legal mistakes or omissions
- Mixed correct and incorrect Austrian law
- Potentially misleading

3 — Mostly correct
- Legally sound under Austrian tax law
- Minor omissions or lack of detail
- No major legal errors

4 — Fully correct / Excellent
- Completely correct under Austrian tax law
- Matches reference answer in legal substance
- Precise and applicable in practice

STRICT RULES:
- Be conservative: if unsure, lower the score
- Any legal inaccuracy cannot be 4
- Any wrong tax rate or threshold max score is 2
- Do not assume or guess tax rules
- Penalize confident but incorrect statements heavily
- Focus only on legal correctness
- A concise legal answer is better than a long one.
- Penalize the model for unnecessary long or introductory sentences. The goal is technical precision, not word count.

OUTPUT FORMAT:
Evaluation: brief legal reasoning referencing Austrian law
Total rating: integer 1–4

Question:
{question}

Reference Answer:
{reference}

System Answer:
{answer}

Begin evaluation.

Evaluation:
"""  # End of prompt template string


def main():  # Define main execution function
    try:  # Start error handling block for file loading
        df_human = pd.read_csv("dataset_clean_fixed.csv", sep = ";")  # Load dataset with human/reference answers
        df_inference = pd.read_csv("inference_results.csv", sep = ";")  # Load model-generated answers

        df_human["id"] = df_human["id"].astype(str)  # Ensure ID column is string type for safe merging

        examples = pd.merge(df_human, df_inference, on="id").copy()  # Merge datasets on shared ID column

        if len(examples) == 0:  # Check if merge produced no rows
            print("Merge failed: No matching IDs")  # Warn user if merge failed
            return  # Exit function early

        print("Data loaded. Merged rows:", len(examples))  # Print number of merged samples

    except Exception as e:  # Catch any file reading or merge errors
        print("File Error:", e)  # Print error message
        return  # Exit program safely

    results = []  # Initialize list to store evaluation results

    print("STARTING LEGAL JUDGE EVALUATION")  # Notify start of evaluation loop

    for i, row in tqdm(examples.iterrows(), total=len(examples)):  # Loop through dataset with progress bar
        try:  # Start try block for API call safety

            prompt_content = JUDGE_PROMPT.format(  # Fill prompt template with row data
                question=row["prompt"],  # Insert question text
                reference=row["correct_answer"],  # Insert reference/legal correct answer
                answer=row["answer"]  # Insert system-generated answer
            )

            output = client.chat.completions.create(  # Call OpenAI chat completion API
                model=model,  # Specify model name
                messages=[  # Provide conversation context
                    {"role": "system", "content": "You are a strict Austrian tax law evaluator."},  # System role instruction
                    {"role": "user", "content": prompt_content}  # User prompt with evaluation task
                ],
                temperature=0.2,  # Low temperature for consistent evaluation
                max_tokens=400  # Limit output length
            )

            response = output.choices[0].message.content.strip()  # Extract and clean model response text

            print(f"\nSAMPLE {i + 1} FEEDBACK")  # Print sample number
            print(response)  # Print full evaluation output

            score_match = re.search(r"Total rating:\s*([1-4])", response)  # Extract numeric score using regex

            if score_match:  # If regex matched a score
                score = int(score_match.group(1))  # Convert extracted score to integer
            else:
                score = None  # If no score found, assign None

            results.append({"id": row["id"], "score": score})  # Store result for this row

            time.sleep(1)  # Pause to avoid rate limits

        except Exception:  # Catch API or runtime errors per row
            print("API Error:")  # Print error header
            traceback.print_exc()  # Print full stack trace for debugging
            results.append({"id": row["id"], "score": None})  # Store failed result

    df_final = pd.DataFrame(results)  # Convert results list into DataFrame
    avg_score = df_final["score"].mean()  # Compute average score across all evaluated samples

    print("\nFINAL AVERAGE SCORE:", round(avg_score, 2) if not pd.isna(avg_score) else "N/A")  # Print final score summary

    df_final.to_csv("judge_evaluation_inf.csv", index=False)  # Save evaluation results to CSV file

if __name__ == "__main__":  # Standard Python entry point check
    main()  # Run main function when script is executed

    """
    FINAL AVERAGE SCORE: 2.34
    """

Data loaded. Merged rows: 643
STARTING LEGAL JUDGE EVALUATION


  0%|          | 0/643 [00:00<?, ?it/s]


SAMPLE 1 FEEDBACK
Evaluation: The system answer correctly identifies the legal basis for the corporate tax assessment as outlined in the Körperschaftsteuergesetz (KStG) and references relevant sections of the Einkommensteuergesetz (EStG) for income determination. However, it introduces unnecessary complexity by discussing provisions related to profit distributions and reserves, which are not directly relevant to the basic definition of the tax base. While the core information is accurate, the additional details could lead to confusion regarding the primary focus of the question. Overall, the answer is mostly correct but contains minor omissions and unnecessary elaboration.

Total rating: 3


  0%|          | 1/643 [00:04<49:39,  4.64s/it]


SAMPLE 2 FEEDBACK
Evaluation: 

1. **Legal correctness**: The system answer correctly identifies the concept of "verdeckte Gewinnausschüttung" (hidden profit distribution) under § 8 Abs. 3 KStG, which is applicable when a corporation grants a zero-interest loan to a shareholder. However, it incorrectly cites § 8 Abs. 3 KStG instead of § 8 Abs. 2 KStG, which is the correct provision for this situation. The explanation regarding the income tax implications for the shareholder is accurate, referencing § 27 Abs. 1 EStG appropriately. The mention of the tax implications for the corporation is also mostly correct, but the reference to the "Unbedenklichkeitsverordnung" (which is not directly relevant here) is misleading.

2. **Jurisdiction accuracy**: The answer is specific to Austrian law, correctly referencing the KStG and EStG, which are applicable in Austria.

3. **Completeness**: The system answer covers the main points related to hidden profit distribution and the tax implications for 

...


SAMPLE 642 FEEDBACK
Evaluation: The system answer contains several inaccuracies regarding the changes to the Grunderwerbsteuer (property transfer tax) in Austria in 2016. While it correctly identifies the introduction of a new method for determining the assessment basis (higher of purchase price or property value), it incorrectly states that the tax rate is now uniformly 3.5% for property purchases, which is misleading as the rate varies based on the transaction type and circumstances. Additionally, the mention of a potential 30% tax rate for share transfers in property-owning companies is exaggerated and not reflective of the actual law. The reference answer correctly identifies the introduction of a tiered rate for family transfers (0.5% to 3.5%), which is not addressed in the system answer. Overall, the system answer contains significant legal inaccuracies and misleading information.

Total rating: 2


100%|█████████▉| 642/643 [52:08<00:05,  5.18s/it]


SAMPLE 643 FEEDBACK
Evaluation: The System Answer provides a broad overview of the Abgabenänderungsgesetz 2014, mentioning changes in various tax laws such as EStG, KStG, and UStG. However, it fails to specifically highlight the key aspects mentioned in the Reference Answer, particularly the tightening of self-disclosure rights and the introduction of the NoVA increase. While the answer is mostly accurate regarding the areas affected by the law, it lacks the precision and focus on the most significant changes, which could lead to misunderstandings about the law's primary impacts. Therefore, it contains minor omissions and lacks the specificity required for a fully correct answer.

Total rating: 3


100%|██████████| 643/643 [52:12<00:00,  4.87s/it]


FINAL AVERAGE SCORE: 2.34


In [2]:
import pandas as pd
from tqdm import tqdm  # Import tqdm for progress bar visualization in loops
import traceback  # Import traceback for detailed error printing
from openai import OpenAI  # Import OpenAI client to call the LLM API
import time
import re  # Import regex module (NOTE: required because you use re.search later)

client = OpenAI(api_key = "api_key_here")  # Initialize OpenAI client with API key (replace "x" with real key)
model = "gpt-4o-mini"  # Define which OpenAI model to use for evaluation

# Define a large prompt template used to evaluate tax answers
JUDGE_PROMPT = """
You are an expert Austrian tax law evaluator.

Your task is to assess how accurately and correctly a system-generated answer responds to a tax law question under Austrian law.

You MUST evaluate strictly based on Austrian tax regulations (e.g. EStG, UStG, BAO etc.). Do not assume general tax knowledge is sufficient.

Inputs:
- TAX QUESTION
- REFERENCE ANSWER (legally correct answer)
- SYSTEM ANSWER (to be evaluated)

Evaluation criteria:

1. Legal correctness
- Must comply with Austrian tax law
- Check tax rates, thresholds, and legal rules

2. Jurisdiction accuracy
- Must be Austria-specific (not Germany or generic EU law)

3. Completeness
- Compare against reference answer coverage

4. Risk of misinformation
- Penalize hallucinated or incorrect legal claims heavily

Grading Scale (ONLY 1–4):

1 — Incorrect / Dangerous
- Legally wrong under Austrian tax law
- Misleading or hallucinated rules
- Wrong jurisdiction or tax rates
- Would lead to incorrect tax decisions

2 — Partially correct
- Some correct elements but important legal mistakes or omissions
- Mixed correct and incorrect Austrian law
- Potentially misleading

3 — Mostly correct
- Legally sound under Austrian tax law
- Minor omissions or lack of detail
- No major legal errors

4 — Fully correct / Excellent
- Completely correct under Austrian tax law
- Matches reference answer in legal substance
- Precise and applicable in practice

STRICT RULES:
- Be conservative: if unsure, lower the score
- Any legal inaccuracy cannot be 4
- Any wrong tax rate or threshold max score is 2
- Do not assume or guess tax rules
- Penalize confident but incorrect statements heavily
- Focus only on legal correctness
- A concise legal answer is better than a long one.
- Penalize the model for unnecessary long or introductory sentences. The goal is technical precision, not word count.

OUTPUT FORMAT:
Evaluation: brief legal reasoning referencing Austrian law
Total rating: integer 1–4

Question:
{question}

Reference Answer:
{reference}

System Answer:
{answer}

Begin evaluation.

Evaluation:
"""  # End of prompt template string


def main():  # Define main execution function
    try:  # Start error handling block for file loading
        df_human = pd.read_csv("dataset_clean_fixed.csv", sep = ";")  # Load dataset with human/reference answers
        df_inference = pd.read_csv("fine_tuned_results.csv", sep = ";")  # Load model-generated answers

        df_human["id"] = df_human["id"].astype(str)  # Ensure ID column is string type for safe merging

        examples = pd.merge(df_human, df_inference, on="id").copy()  # Merge datasets on shared ID column

        if len(examples) == 0:  # Check if merge produced no rows
            print("Merge failed: No matching IDs")  # Warn user if merge failed
            return  # Exit function early

        print("Data loaded. Merged rows:", len(examples))  # Print number of merged samples

    except Exception as e:  # Catch any file reading or merge errors
        print("File Error:", e)  # Print error message
        return  # Exit program safely

    results = []  # Initialize list to store evaluation results

    print("STARTING LEGAL JUDGE EVALUATION") # Notify start of evaluation loop

    for i, row in tqdm(examples.iterrows(), total=len(examples)): # Loop through dataset with progress bar
        try:  # Start try block for API call safety

            prompt_content = JUDGE_PROMPT.format(  # Fill prompt template with row data
                question=row["prompt"],  # Insert question text
                reference=row["correct_answer"],  # Insert reference/legal correct answer
                answer=row["answer"]  # Insert system-generated answer
            )

            output = client.chat.completions.create(  # Call OpenAI chat completion API
                model=model,  # Specify model name
                messages=[  # Provide conversation context
                    {"role": "system", "content": "You are a strict Austrian tax law evaluator."},  # System role instruction
                    {"role": "user", "content": prompt_content}  # User prompt with evaluation task
                ],
                temperature=0.2,  # Low temperature for consistent evaluation
                max_tokens=400  # Limit output length
            )

            response = output.choices[0].message.content.strip()  # Extract and clean model response text

            print(f"\nSAMPLE {i + 1} FEEDBACK")  # Print sample number
            print(response)  # Print full evaluation output

            score_match = re.search(r"Total rating:\s*([1-4])", response)  # Extract numeric score using regex

            if score_match:  # If regex matched a score
                score = int(score_match.group(1))  # Convert extracted score to integer
            else:
                score = None  # If no score found, assign None

            results.append({"id": row["id"], "score": score})  # Store result for this row

            time.sleep(1)  # Pause to avoid rate limits

        except Exception:  # Catch API or runtime errors per row
            print("API Error:")  # Print error header
            traceback.print_exc()  # Print full stack trace for debugging
            results.append({"id": row["id"], "score": None})  # Store failed result

    df_final = pd.DataFrame(results)  # Convert results list into DataFrame
    avg_score = df_final["score"].mean()  # Compute average score across all evaluated samples

    print("\nFINAL AVERAGE SCORE:", round(avg_score, 2) if not pd.isna(avg_score) else "N/A")  # Print final score summary

    df_final.to_csv("judge_evaluation_ft.csv", index=False)  # Save evaluation results to CSV file

if __name__ == "__main__":  # Standard Python entry point check
    main()  # Run main function when script is executed

"""
FINAL AVERAGE SCORE: 1.5
"""

Data loaded. Merged rows: 643
STARTING LEGAL JUDGE EVALUATION


  0%|          | 0/643 [00:00<?, ?it/s]


SAMPLE 1 FEEDBACK
Evaluation: The system answer correctly identifies the basis for the corporate income tax (Körperschaftsteuer) as the income or profit according to § 7 KStG. However, it introduces a reference to § 1 Abs. 1 KStG, which is not necessary for defining the tax base and could lead to confusion about the specific provisions relevant to the assessment. The reference answer is more precise in stating the unbeschränkte Steuerpflicht, which is a critical aspect of the tax base. Therefore, while the system answer is mostly correct, it lacks the clarity and specificity of the reference answer.

Total rating: 3


  0%|          | 1/643 [00:03<36:49,  3.44s/it]


SAMPLE 2 FEEDBACK
Evaluation: The system answer incorrectly interprets the tax implications of a zero-interest loan from a corporation to its shareholder. It fails to recognize that the non-charged interest constitutes a hidden distribution (verdeckte Ausschüttung) under § 8 Abs. 2 KStG, which would increase the taxable income. The reference answer correctly identifies this aspect and cites relevant sections of the Körperschaftsteuergesetz (KStG). The system answer also misapplies the concept of a taxable exchange of services, which is not applicable in this context. Therefore, it contains significant legal inaccuracies.

Total rating: 1


...


SAMPLE 642 FEEDBACK
Evaluation: The system answer incorrectly references the UStG (Value Added Tax Act) instead of the Grunderwerbsteuergesetz (Property Transfer Tax Act) relevant to the question about the changes in property transfer tax in 2016. The reference to the assessment basis being determined by the value of the property is misleading and does not address the introduction of the tiered rate for family transfers. This misalignment with the correct legal framework and the omission of key details regarding the tiered rate leads to significant inaccuracies.

Total rating: 1


100%|█████████▉| 642/643 [40:02<00:03,  3.28s/it]


SAMPLE 643 FEEDBACK
Evaluation: The System Answer mentions "steuerliche Neuerungen und Änderungen," which is vague and does not specifically address the key aspects of the Abgabenänderungsgesetz 2014 as outlined in the Reference Answer. It incorrectly focuses on the "Stichtagsregimes" and "Gewinnfreibetrags," which are not the primary changes highlighted in the Reference Answer. The reference to § 1 AbgÄG 2014 is also misleading as it does not accurately reflect the main changes, such as the tightening of self-disclosure rights and the introduction of the NoVA increase. Therefore, the System Answer contains inaccuracies and lacks the necessary specificity regarding the law.

Total rating: 2


100%|██████████| 643/643 [40:06<00:00,  3.74s/it]


FINAL AVERAGE SCORE: 1.5


In [3]:
import pandas as pd
import time
from tqdm import tqdm  # Import tqdm for progress bar visualization in loops
import traceback  # Import traceback for detailed error printing
from openai import OpenAI  # Import OpenAI client to call the LLM API
import re  # Import regex module (NOTE: required because you use re.search later)

client = OpenAI(api_key = "api_key_here")  # Initialize OpenAI client with API key (replace "x" with real key)
model = "gpt-4o-mini"  # Define which OpenAI model to use for evaluation

# Define a large prompt template used to evaluate tax answers

JUDGE_PROMPT = """
You are an expert Austrian tax law evaluator.

Your task is to assess how accurately and correctly a system-generated answer responds to a tax law question under Austrian law.

You MUST evaluate strictly based on Austrian tax regulations (e.g. EStG, UStG, BAO etc.). Do not assume general tax knowledge is sufficient.

Inputs:
- TAX QUESTION
- REFERENCE ANSWER (legally correct answer)
- SYSTEM ANSWER (to be evaluated)

Evaluation criteria:

1. Legal correctness
- Must comply with Austrian tax law
- Check tax rates, thresholds, and legal rules

2. Jurisdiction accuracy
- Must be Austria-specific (not Germany or generic EU law)

3. Completeness
- Compare against reference answer coverage

4. Risk of misinformation
- Penalize hallucinated or incorrect legal claims heavily

Grading Scale (ONLY 1–4):

1 — Incorrect / Dangerous
- Legally wrong under Austrian tax law
- Misleading or hallucinated rules
- Wrong jurisdiction or tax rates
- Would lead to incorrect tax decisions

2 — Partially correct
- Some correct elements but important legal mistakes or omissions
- Mixed correct and incorrect Austrian law
- Potentially misleading

3 — Mostly correct
- Legally sound under Austrian tax law
- Minor omissions or lack of detail
- No major legal errors

4 — Fully correct / Excellent
- Completely correct under Austrian tax law
- Matches reference answer in legal substance
- Precise and applicable in practice

STRICT RULES:
- Be conservative: if unsure, lower the score
- Any legal inaccuracy cannot be 4
- Any wrong tax rate or threshold max score is 2
- Do not assume or guess tax rules
- Penalize confident but incorrect statements heavily
- Focus only on legal correctness
- A concise legal answer is better than a long one
- Penalize the model for unnecessary long or introductory sentences. The goal is technical precision, not word count.

OUTPUT FORMAT:
Evaluation: brief legal reasoning referencing Austrian law
Total rating: integer 1–4

Question:
{question}

Reference Answer:
{reference}

System Answer:
{answer}

Begin evaluation.

Evaluation:
"""  # End of prompt template string


def main():  # Define main execution function
    try:  # Start error handling block for file loading
        df_human = pd.read_csv("dataset_clean_fixed.csv", sep = ";")  # Load dataset with human/reference answers
        df_inference = pd.read_csv("rag_results.csv", sep = ";")  # Load model-generated answers

        df_human["id"] = df_human["id"].astype(str)  # Ensure ID column is string type for safe merging

        examples = pd.merge(df_human, df_inference, on="id").copy()  # Merge datasets on shared ID column

        if len(examples) == 0:  # Check if merge produced no rows
            print("Merge failed: No matching IDs")  # Warn user if merge failed
            return  # Exit function early

        print("Data loaded. Merged rows:", len(examples))  # Print number of merged samples

    except Exception as e:  # Catch any file reading or merge errors
        print("File Error:", e)  # Print error message
        return  # Exit program safely

    results = []  # Initialize list to store evaluation results

    print("STARTING LEGAL JUDGE EVALUATION")  # Notify start of evaluation loop

    for i, row in tqdm(examples.iterrows(), total=len(examples)):  # Loop through dataset with progress bar
        try:  # Start try block for API call safety

            prompt_content = JUDGE_PROMPT.format(  # Fill prompt template with row data
                question=row["prompt"],  # Insert question text
                reference=row["correct_answer"],  # Insert reference/legal correct answer
                answer=row["answer"]  # Insert system-generated answer
            )

            output = client.chat.completions.create(  # Call OpenAI chat completion API
                model=model,  # Specify model name
                messages=[  # Provide conversation context
                    {"role": "system", "content": "You are a strict Austrian tax law evaluator."},  # System role instruction
                    {"role": "user", "content": prompt_content}  # User prompt with evaluation task
                ],
                temperature=0.2,  # Low temperature for consistent evaluation
                max_tokens=400  # Limit output length
            )

            response = output.choices[0].message.content.strip()  # Extract and clean model response text

            print(f"\nSAMPLE {i + 1} FEEDBACK")  # Print sample number
            print(response)  # Print full evaluation output

            score_match = re.search(r"Total rating:\s*([1-4])", response)  # Extract numeric score using regex

            if score_match:  # If regex matched a score
                score = int(score_match.group(1))  # Convert extracted score to integer
            else:
                score = None  # If no score found, assign None

            results.append({"id": row["id"], "score": score})  # Store result for this row

            time.sleep(1)  # Pause to avoid rate limits

        except Exception:  # Catch API or runtime errors per row
            print("API Error:")  # Print error header
            traceback.print_exc()  # Print full stack trace for debugging
            results.append({"id": row["id"], "score": None})  # Store failed result

    df_final = pd.DataFrame(results)  # Convert results list into DataFrame
    avg_score = df_final["score"].mean()  # Compute average score across all evaluated samples

    print("\nFINAL AVERAGE SCORE:", round(avg_score, 2) if not pd.isna(avg_score) else "N/A")  # Print final score summary

    df_final.to_csv("judge_evaluation_rag.csv", index=False)  # Save evaluation results to CSV file

if __name__ == "__main__":  # Standard Python entry point check
    main()  # Run main function when script is executed

"""
FINAL AVERAGE SCORE: 1.33
"""

Data loaded. Merged rows: 643
STARTING LEGAL JUDGE EVALUATION


  0%|          | 0/643 [00:00<?, ?it/s]


SAMPLE 1 FEEDBACK
Evaluation: The system answer incorrectly states that the provided information is not relevant for determining the corporate income tax (Körperschaftsteuer). According to § 7 Abs. 1 KStG, the taxable income of a corporation is indeed the basis for calculating the Körperschaftsteuer. This fundamental legal principle is ignored in the system answer, leading to a significant misinformation risk.

Total rating: 1


  0%|          | 1/643 [00:03<35:26,  3.31s/it]


SAMPLE 2 FEEDBACK
Evaluation: The system answer is legally incorrect as it states that there are no tax regulations regarding interest-free loans to shareholders. In fact, according to § 8 Abs. 2 KStG, the non-accounted interest is considered a hidden distribution, which affects the income calculation. This is a significant omission and misrepresentation of Austrian tax law. 

Total rating: 1


...


SAMPLE 642 FEEDBACK
Evaluation: The system answer incorrectly states that there were no changes to the Grunderwerbsteuer in 2016, which contradicts the reference answer that highlights the introduction of a tiered rate for transfers within families. This is a significant legal error as it misrepresents the actual changes in Austrian tax law. Therefore, the answer is misleading and could lead to incorrect tax decisions.

Total rating: 1


100%|█████████▉| 642/643 [40:04<00:03,  3.23s/it]


SAMPLE 643 FEEDBACK
Evaluation: The System Answer inaccurately describes the Abgabenänderungsgesetz 2014. It fails to mention the specific changes regarding the self-disclosure rights (Selbstanzeige-Recht) and the introduction of the NoVA increase, which are key aspects of the law. Additionally, the phrasing "Berechtigung der Abgabenänderungen" is vague and does not accurately reflect the legal content of the law. The answer lacks precision and contains misleading elements about the nature of the law.

Total rating: 2


100%|██████████| 643/643 [40:07<00:00,  3.74s/it]


FINAL AVERAGE SCORE: 1.33
